# Recreate all performance metrics with 95% intervals

This notebook produces the sole input table for `compare_all_estimators.ipynb`. Bayesian estimators use 95% HDIs; MLE uses its existing 95% Wald confidence intervals. The workflow refuses to write a master table unless all nine configurations are available for every estimator.


In [1]:
from recreate_metrics_95ci import input_audit

audit = input_audit()
display(audit)

missing = audit.query("status == 'missing'")
if not missing.empty:
    print("Missing posterior traces. Do not substitute the old 94% summary bounds.")


,estimator,J,N,P,status,source_file
0,TURBOLFI,100,10000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
1,TURBOLFI,100,20000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
2,TURBOLFI,100,50000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
3,TURBOLFI,200,10000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
4,TURBOLFI,200,20000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
5,TURBOLFI,200,50000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
6,TURBOLFI,500,10000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
7,TURBOLFI,500,20000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
8,TURBOLFI,500,50000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...
9,AdaptiveSampler,100,10000,10,available,/Users/preto/Documents/EPFL/PhD/transp-or/TooM...


## Required TURBOLFI input

Set `RECOVER_MISSING_TURBOLFI_TRACES = True` in the next cell to restore each saved GP checkpoint and rerun its MCMC stage automatically. This does not retrain BOLFI, but it does run nine MCMC jobs. The `.pkl` file restores the GP; the resulting NetCDF posterior draws are then used for the exact 95% HDIs.


In [2]:
from recreate_metrics_95ci import run

# Set to True to restore the nine missing TURBOLFI traces from their .pkl checkpoints.
RECOVER_MISSING_TURBOLFI_TRACES = False

if missing.empty or RECOVER_MISSING_TURBOLFI_TRACES:
    output_file = run(recover_turbolfi=RECOVER_MISSING_TURBOLFI_TRACES)
    print(output_file)
else:
    output_file = None
    print("Set RECOVER_MISSING_TURBOLFI_TRACES = True, then rerun this cell to create the complete 95% master CSV.")


Ignoring non-parameter posterior variables in restaurants_mnl_P10_J100_N10k_I0_C0_M0_woJIT~01.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J100_N20k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J100_N50k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J200_N10k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J200_N20k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J200_N50k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J500_N10k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J500_N20k_I0_C0_M0_woJIT.nc: ['log_like']
Ignoring non-parameter posterior variables in restaurants_mnl_P10_J500_N50k_I0_C0_M0_woJIT.nc: ['log_like']
Wrote 360 validated rows 

## Validation guarantees

- Only the ten model coefficients are retained; diagnostics such as `log_like` are excluded.
- Adaptive Sampler, BI, and TURBOLFI HDIs use `arviz.summary(..., hdi_prob=0.95)`.
- Only BI is harmonized to 500 post-warmup draws per chain (2,000 total); Adaptive Sampler and TURBOLFI retain their complete configured traces.
- The original source draw count, tuning steps, sampling time, and recorded total run time are retained in the output columns.
- MLE is checked to be exactly ±1.96 standard errors.
- The master table must contain 360 unique rows: 4 estimators × 9 configurations × 10 parameters.
